# Step 2 — Color–Magnitude Diagram (non-RUWE colored)
#
# Purpose
#   - Load a final member catalog (CSV) produced by Step 1.
#   - Clean CMD columns (bp_rp, phot_g_mean_mag) and optionally apply quality cuts.
#   - Plot a standard Gaia CMD (G vs. BP-RP) with a clean, publication-ready style.
#
# Notes
#   - Gaia "phot_g_mean_mag" is an apparent G magnitude unless you already converted it.
#     If you want absolute magnitude, provide a distance modulus or use parallax (with care).
#   - This step intentionally does not color-code by RUWE; that will be in a later step.

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
@dataclass
class CMDConfig:
    cluster_name: str
    input_members_csv: str

    # Column names (expected in your members CSV)
    color_col: str = "bp_rp"
    mag_col: str = "phot_g_mean_mag"

    # Optional: compute absolute magnitude using a fixed distance modulus (m - M)
    # Set to None to plot the magnitude column as-is.
    distance_modulus: Optional[float] = None

    # Optional: apply a magnitude range cut (recommended for cleaner plots)
    mag_limits: Optional[Tuple[float, float]] = None  # e.g., (8, 20)

    # Plot styling
    figsize: Tuple[float, float] = (8.0, 8.0)
    marker_size: float = 2.0
    alpha: float = 0.6

    # Output (optional)
    save_path: Optional[str] = None  # e.g., "M67_CMD.png"
    dpi: int = 300


# ---------------------------------------------------------------------
# Core functionality
# ---------------------------------------------------------------------
def load_members(path: str) -> pd.DataFrame:
    path_obj = Path(path)
    if not path_obj.exists():
        raise FileNotFoundError(f"Members file not found: {path_obj.resolve()}")
    return pd.read_csv(path_obj)


def prepare_cmd_data(df: pd.DataFrame, cfg: CMDConfig) -> pd.DataFrame:
    df = df.copy()

    # Coerce CMD columns to numeric
    df[cfg.color_col] = pd.to_numeric(df[cfg.color_col], errors="coerce")
    df[cfg.mag_col] = pd.to_numeric(df[cfg.mag_col], errors="coerce")

    # Drop NaNs
    cmd = df.dropna(subset=[cfg.color_col, cfg.mag_col]).copy()

    # Optional magnitude cut
    if cfg.mag_limits is not None:
        mag_min, mag_max = cfg.mag_limits
        cmd = cmd[(cmd[cfg.mag_col] >= mag_min) & (cmd[cfg.mag_col] <= mag_max)].copy()

    # Optional absolute magnitude conversion using a fixed distance modulus
    if cfg.distance_modulus is not None:
        cmd["abs_mag"] = cmd[cfg.mag_col] - float(cfg.distance_modulus)
    else:
        cmd["abs_mag"] = cmd[cfg.mag_col]

    return cmd


def plot_cmd(cmd: pd.DataFrame, cfg: CMDConfig) -> None:
    plt.figure(figsize=cfg.figsize)

    plt.scatter(
        cmd[cfg.color_col].values,
        cmd["abs_mag"].values,
        s=cfg.marker_size,
        alpha=cfg.alpha,
        label="Members",
    )

    plt.gca().invert_yaxis()
    plt.xlabel(r"Color ($G_{\rm BP} - G_{\rm RP}$)")
    ylabel = r"Absolute Magnitude ($G$)" if cfg.distance_modulus is not None else r"Apparent Magnitude ($G$)"
    plt.ylabel(ylabel)
    plt.title(f"{cfg.cluster_name} Color–Magnitude Diagram (Final Members)")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="best", frameon=True)
    plt.tight_layout()

    if cfg.save_path is not None:
        out_path = Path(cfg.save_path)
        plt.savefig(out_path, dpi=cfg.dpi)
        print(f"Saved CMD figure: {out_path.resolve()}")

    plt.show()


def main(cfg: CMDConfig) -> None:
    df = load_members(cfg.input_members_csv)
    print(f"Loaded members: {len(df)} rows")

    cmd = prepare_cmd_data(df, cfg)
    print(f"CMD-ready sample: {len(cmd)} rows")

    plot_cmd(cmd, cfg)


# ---------------------------------------------------------------------
# User configuration (edit here)
# ---------------------------------------------------------------------
cfg = CMDConfig(
    cluster_name="M67",
    input_members_csv="M67_Members_Final.csv",
    distance_modulus=None,           # set numeric value if you want absolute magnitude via DM
    mag_limits=None,                 # e.g., (8, 20) to trim extremes
    save_path=None,                  # e.g., "M67_CMD.png"
)

main(cfg)

